<a href="https://colab.research.google.com/github/ankush284/Email_Classification/blob/main/Email_Spam_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning BERT for Email Classification — PyTorch (Google Colab)

We teach a pretrained BERT model to classify mails into **2 classes**:
ham, spam.

**Fine-tuning in one line:** BERT already understands English. We add a small
output layer and train it a bit more on our labelled mails.

**New in this version:** a light **text preprocessing** step for mails.

**First: turn ON the GPU** -> Runtime -> Change runtime type -> GPU


## Step 1 — Install libraries

In [ ]:
!pip install -q transformers datasets accelerate
  # HF libs + GPU training helper
  # to dowload any model or dataset from the HF

## Step 2 — Upload the dataset

Click **Choose Files** and pick your `email.csv`.

In [ ]:
from google.colab import files          # Colab file upload helper
uploaded = files.upload()               # opens a file picker; choose Tweets.csv

Saving email.csv to email (1).csv


## Step 3 — Imports

In [ ]:
import re, torch, numpy as np, pandas as pd                              # utils + PyTorch #preprocessing
from datasets import Dataset                                             # HF dataset #used for compatibility
from sklearn.model_selection import train_test_split                     # data split
from sklearn.metrics import accuracy_score, classification_report        # scoring
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)                    # BERT + Trainer
# auto token
# sequenceclassification - means for classification task
# trainingarguments - training parameters using which we can train our model
# trainer - uses the training arguments and train the model

## Step 4 — Load the data

Read the CSV and keep only the Message + Category columns.

In [ ]:
df = pd.read_csv("email.csv")            # load the CSV into a DataFrame
df = df[["Message", "Category"]].dropna()   # keep 2 columns, drop empty rows
#df=df.sample(frac=1, ignore_index=True)   # shuffle data
#df=df.head(5000)

print("Shape:", df.shape)                 # rows, columns
df.head()                              # peek at first 5 rows

Shape: (5573, 2)


,Message,Category
0,"Go until jurong point, crazy.. Available only ...",ham
1,Ok lar... Joking wif u oni...,ham
2,Free entry in 2 a wkly comp to win FA Cup fina...,spam
3,U dun say so early hor... U c already then say...,ham
4,"Nah I don't think he goes to usf, he lives aro...",ham


## Step 5 — Text preprocessing (light, for mails)

For BERT we clean **gently** — we do NOT remove stopwords or lemmatize, because
BERT understands grammar and word order. We only remove mail noise: links,
@mentions, the # symbol, odd characters, and extra spaces.

In [ ]:
def clean_text(text):
    text = str(text).lower()                       # convert to string and lowercase
    text = re.sub(r"[^a-z\s.,!?']", " ", text)     # remove anything except letters and basic punctuation
    text = re.sub(r"\s+", " ", text)               # collapse multiple spaces into one
    return text.strip()                            # remove spaces at the start and end

df["Message"] = df["Message"].apply(clean_text)          # clean every row
df.head()


#'''
#re.sub( WHAT_TO_FIND , REPLACE_WITH , WHERE_TO_LOOK )
#re.sub(    r"\s+"    ,     " "      ,     text       )
#'''

,Message,Category
0,"go until jurong point, crazy.. available only ...",ham
1,ok lar... joking wif u oni...,ham
2,free entry in a wkly comp to win fa cup final ...,spam
3,u dun say so early hor... u c already then say...,ham
4,"nah i don't think he goes to usf, he lives aro...",ham


## Step 6 — Convert labels to numbers

BERT needs 0/1 instead of words.

In [ ]:
# the two maps, written out plainly
label2id = {"ham": 0, "spam": 1}   # word  -> number
id2label = {0: "ham", 1: "spam"}   # number -> word

# turn the category words into numbers
df["label"] = df["Category"].map(label2id)               # words -> 0/1

print(id2label)
df["Category"].value_counts()

print(df["label"].isna().sum())
df[df["label"].isna()]

df = df.dropna(subset=["label"])

print(df["label"].isna().sum())
df[df["label"].isna()]

{0: 'ham', 1: 'spam'}
0
0


,Message,Category,label


## Step 7 — split



In [ ]:
train_df, test_df = train_test_split(         # split into train and test
    df[["Message", "label"]],                    # use ALL rows; only need text + label
    test_size=0.2,                            # 20% goes to test
    random_state=42,                          # reproducible split #randomly splits the data
    stratify=df["label"],                     # keep class ratio in both sets
)
print("Train:", len(train_df), "Test:", len(test_df))

Train: 4457 Test: 1115


## Step 8 — Tokenize

Turn each mail into token IDs; pad/cut to length 128.

In [ ]:
model_name = "distilbert-base-uncased"                       # which BERT to use
tokenizer = AutoTokenizer.from_pretrained(model_name)  # load its tokenizer

def tokenize(batch):                                   # function applied to data
    return tokenizer(batch["Message"],                    # the message column
                     padding="max_length",             # pad short message
                     truncation=True,                  # cut long message
                     max_length=128)                   # fixed length = 128 tokens

train_ds = Dataset.from_pandas(train_df).map(tokenize, batched=True)  # tokenize train
test_ds  = Dataset.from_pandas(test_df).map(tokenize, batched=True)  # tokenize test

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/4457 [00:00<?, ? examples/s]

Map:   0%|          | 0/1115 [00:00<?, ? examples/s]

In [ ]:
train_ds

Dataset({
    features: ['Message', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4457
})

In [ ]:
print(train_ds[0]["Message"])
print(train_ds[3]["input_ids"])
print(train_ds[2]["input_ids"])
print(train_ds[3]["token_type_ids"])
print(train_ds[3]["attention_mask"])



he will, you guys close?
[101, 1045, 1005, 2222, 2156, 1010, 2021, 4013, 9215, 3398, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[101, 7929, 1047, 1012, 1012, 5034, 2100, 1045, 14161, 2860, 9033, 3567, 1012, 1012, 11937, 3215, 1061, 1045, 3198, 2094, 1012, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

## Step 9 — Load BERT with a 2-class head

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(  # BERT + classifier on top
    model_name,            # same distil-bert-uncased
    num_labels=2,          # 2 classes: ham / spam
    id2label=id2label,     # readable output labels
    label2id=label2id,     # reverse map
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Step 10 — Accuracy metric

In [ ]:
def compute_metrics(eval_pred):                       # Trainer calls this automatically during evaluation
    logits, labels = eval_pred                        # unpack: logits = raw model scores, labels = true answers
    preds = np.argmax(logits, axis=-1)                # pick the class with the highest score = prediction
    accuracy = accuracy_score(labels, preds)          # compare predictions vs true labels to get accuracy
    return {"accuracy": accuracy}                      # return as a dict so Trainer can log it by name

## Step 11 — Training settings

In [ ]:
args = TrainingArguments(
    output_dir="results",                 # where checkpoints & logs are saved (required)
    eval_strategy="epoch",                # evaluate after every epoch
    save_strategy="epoch",                # save a checkpoint after every epoch
    num_train_epochs=3,                   # full passes through the training data
    per_device_train_batch_size=8,        # training samples per batch
    per_device_eval_batch_size=8,         # evaluation samples per batch
    learning_rate=2e-5,                   # small step size for fine-tuning
    weight_decay=0.01,                    # regularization to reduce overfitting
    logging_steps=50,                     # log metrics every 50 steps
    load_best_model_at_end=True,          # reload the best checkpoint after training
    metric_for_best_model="accuracy",     # judge "best" by accuracy
)

## Step 12 — Train the model

In [ ]:
trainer = Trainer(                         # bundles everything together
    model=model,                           # the BERT model
    args=args,                             # the settings above
    train_dataset=train_ds,                # data to learn from
    eval_dataset=test_ds,                  # data to check on
    compute_metrics=compute_metrics,       # how to score
)

trainer.train()                            # <-- run the actual training loop

Epoch,Training Loss,Validation Loss,Accuracy
1,0.074014,0.058184,0.987444
2,0.000832,0.051835,0.990135
3,0.001454,0.048925,0.991031


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1674, training_loss=0.04763703135019135, metrics={'train_runtime': 197.4715, 'train_samples_per_second': 67.711, 'train_steps_per_second': 8.477, 'total_flos': 442805396857344.0, 'train_loss': 0.04763703135019135, 'epoch': 3.0})

## Step 13 — Evaluate

In [ ]:
preds = trainer.predict(test_ds)                 # predict on the test set
y_pred = np.argmax(preds.predictions, axis=-1)   # chosen class per tweet
y_true = preds.label_ids                         # the true classes

print(classification_report(                     # precision/recall/F1 per class
    y_true, y_pred, target_names=["ham","spam"]
))

              precision    recall  f1-score   support

         ham       0.99      1.00      0.99       966
        spam       0.98      0.95      0.97       149

    accuracy                           0.99      1115
   macro avg       0.99      0.97      0.98      1115
weighted avg       0.99      0.99      0.99      1115



## Step 14 — Test on your own sentences

In [ ]:
from transformers import pipeline                 # easy inference wrapper

clf = pipeline("text-classification",             # task
               model=model, tokenizer=tokenizer)  # use our fine-tuned model

print(clf(clean_text("Hi team, please find the Q2 project update attached. Let's discuss on Thursday at 10 AM.")))     # ham clean then predict
print(clf(clean_text("URGENT: Your account has been locked. Click here to verify your password immediately.")))  # spam example
print(clf(clean_text("A sign-in was detected on your account from a new device. Was this you?")))     # ham example
print(clf(clean_text("You have won $1,000,000 in our international lottery! Send your bank details to claim.")))   # spam example

[{'label': 'ham', 'score': 0.9418354630470276}]
[{'label': 'spam', 'score': 0.9506785869598389}]
[{'label': 'ham', 'score': 0.9748647809028625}]
[{'label': 'spam', 'score': 0.9969300627708435}]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 15 (optional) — Save the model

In [ ]:
trainer.save_model("bert-email-classification")          # save model weights
tokenizer.save_pretrained("bert-email-classification")   # save tokenizer

# from google.colab import drive                      # to save in Google Drive:
# drive.mount("/content/drive")                        # mount your Drive
trainer.save_model("/content/drive/MyDrive/bert-email-classification")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]